# Model Training

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/01 22:36:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/01 22:36:43 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/01 22:36:43 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/08/01 22:36:43 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/08/01 22:36:43 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.


In [3]:
df = spark.read.parquet("../outputs/cleaned_timetable_parquet")

In [4]:
df.show(5, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+----------+
|creation_time      |filename                                                                                                |line_name|modified_time      |number_of_journeys|number_of_stops|operator  |route_description                                 |service_code |route_size|
+-------------------+--------------------------------------------------------------------------------------------------------+---------+-------------------+------------------+---------------+----------+--------------------------------------------------+-------------+----------+
|2022-08-19 12:05:39|X2-None--SCMY-CZ-2026-03-29-Chester_July_2026_(SchOut)__SCMY_PC1033334_224_20260719-BODS_V1_1.xml       |2        |2026-07-02 13:55:43|141    

## Preparing the Target Variable

In [5]:
from pyspark.sql.functions import when

df = df.withColumn(
    "target",
    when(col("route_size") == "Long", 1).otherwise(0)
)

In [6]:
df.select(
    "route_size",
    "target"
).show(10)

+----------+------+
|route_size|target|
+----------+------+
|      Long|     1|
|     Short|     0|
|    Medium|     0|
|     Short|     0|
|    Medium|     0|
|     Short|     0|
|     Short|     0|
|    Medium|     0|
|     Short|     0|
|     Short|     0|
+----------+------+
only showing top 10 rows


## Feature Preparation

In [7]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

In [8]:
indexer = StringIndexer(
    inputCol="operator",
    outputCol="operator_index"
)

df = indexer.fit(df).transform(df)

In [9]:
assembler = VectorAssembler(
    inputCols=[
        "number_of_stops",
        "number_of_journeys",
        "operator_index"
    ],
    outputCol="features"
)

dataset = assembler.transform(df)

## Train-Test Split

In [10]:
train_data, test_data = dataset.randomSplit([0.8, 0.2], seed=42)

print("Training records:", train_data.count())
print("Testing records:", test_data.count())

Training records: 99
Testing records: 24


## Train Decision Tree Classifier

In [11]:
from pyspark.ml.classification import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="target"
)

model = dt.fit(train_data)

26/08/01 22:39:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## Make Predictions

In [12]:
predictions = model.transform(test_data)

In [13]:
predictions.select(
    "route_size",
    "target",
    "prediction",
    "probability"
).show(10, truncate=False)

+----------+------+----------+-----------+
|route_size|target|prediction|probability|
+----------+------+----------+-----------+
|Short     |0     |0.0       |[1.0,0.0]  |
|Short     |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
|Long      |1     |1.0       |[0.0,1.0]  |
|Medium    |0     |0.0       |[1.0,0.0]  |
+----------+------+----------+-----------+
only showing top 10 rows


## Feature Importance

In [14]:
print("Feature Importances:")
print(model.featureImportances)

Feature Importances:
(3,[0,1],[0.9409660107334527,0.05903398926654745])


## Save Trained Model

In [15]:
model.write().overwrite().save("../models/decision_tree_model")

print("Decision Tree model saved successfully!")

Decision Tree model saved successfully!


# Summary

In this notebook, the cleaned dataset was prepared for machine learning by creating a target variable and assembling numerical features. A Decision Tree classifier was trained using the training dataset, predictions were generated for the testing dataset, feature importance was examined, and the trained model was saved for evaluation.